<a href="https://colab.research.google.com/github/Ebrar6565/smart-city-satellite-analysis/blob/main/notebooks%20/03_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U-Net ile Semantic Segmentation Modelinin Eğitilmesi

Bu notebook içerisinde LoveDA eğitim verileri kullanılarak ilk semantic segmentation modeli oluşturulacaktır.

Gerçekleştirilecek işlemler:

- LoveDA Train verisinin hazırlanması
- Urban ve Rural eğitim örneklerinin seçilmesi
- Eğitim ve validation veri yükleyicilerinin oluşturulması
- U-Net modelinin kurulması
- Loss fonksiyonu ve optimizasyon algoritmasının tanımlanması
- Küçük veri üzerinde çalışma kontrolü
- Modelin eğitilmesi
- Eğitim ve validation loss değerlerinin incelenmesi
- Tahmin maskelerinin görselleştirilmesi

Google Drive'da bulunan 40 validation örneği modelin eğitimi için değil, modelin görmediği görüntüler üzerindeki performansını incelemek için kullanılacaktır.

In [ ]:
import os
import random
import shutil
import zipfile

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    ConcatDataset
)

print("PyTorch sürümü:", torch.__version__)
print("CUDA kullanılabilir mi?:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Kullanılacak cihaz:", device)

PyTorch sürümü: 2.11.0+cpu
CUDA kullanılabilir mi?: False
Kullanılacak cihaz: cpu


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
train_zip_path = "/content/Train.zip"

if os.path.exists(train_zip_path):
    print("Train.zip zaten bulunuyor.")
else:
    print("LoveDA Train.zip indiriliyor...")

LoveDA Train.zip indiriliyor...


In [ ]:
!wget -O /content/Train.zip "https://zenodo.org/records/5706578/files/Train.zip?download=1"

--2026-07-18 11:26:41--  https://zenodo.org/records/5706578/files/Train.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.185.48.75, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4021669263 (3.7G) [application/octet-stream]
Saving to: ‘/content/Train.zip’

/content/Train.zip  100%[===================>]   3.75G  5.88MB/s    in 10m 36s 

2026-07-18 11:37:18 (6.03 MB/s) - ‘/content/Train.zip’ saved [4021669263/4021669263]



In [ ]:
from pathlib import Path

train_zip_path = Path("/content/Train.zip")

if train_zip_path.exists():
    file_size_gb = train_zip_path.stat().st_size / (1024 ** 3)

    print("Train.zip başarıyla indirildi.")
    print(f"Dosya boyutu: {file_size_gb:.2f} GB")
else:
    print("Train.zip bulunamadı.")

Train.zip başarıyla indirildi.
Dosya boyutu: 3.75 GB


In [ ]:
import zipfile

train_zip_path = "/content/Train.zip"

with zipfile.ZipFile(train_zip_path, "r") as zip_file:
    file_names = zip_file.namelist()

print("ZIP içerisindeki toplam kayıt sayısı:", len(file_names))

print("\nİlk 30 kayıt:")
for file_name in file_names[:30]:
    print(file_name)

ZIP içerisindeki toplam kayıt sayısı: 5051

İlk 30 kayıt:
Train/
Train/Rural/
Train/Rural/images_png/
Train/Rural/images_png/0.png
Train/Rural/images_png/1.png
Train/Rural/images_png/10.png
Train/Rural/images_png/100.png
Train/Rural/images_png/1000.png
Train/Rural/images_png/1001.png
Train/Rural/images_png/1002.png
Train/Rural/images_png/1003.png
Train/Rural/images_png/1004.png
Train/Rural/images_png/1005.png
Train/Rural/images_png/1006.png
Train/Rural/images_png/1007.png
Train/Rural/images_png/1008.png
Train/Rural/images_png/1009.png
Train/Rural/images_png/101.png
Train/Rural/images_png/1010.png
Train/Rural/images_png/1011.png
Train/Rural/images_png/1012.png
Train/Rural/images_png/1013.png
Train/Rural/images_png/1014.png
Train/Rural/images_png/1015.png
Train/Rural/images_png/1016.png
Train/Rural/images_png/1017.png
Train/Rural/images_png/1018.png
Train/Rural/images_png/1019.png
Train/Rural/images_png/102.png
Train/Rural/images_png/1020.png


In [ ]:
import json
import random
import shutil
import zipfile

from pathlib import Path


TRAIN_SAMPLE_COUNT_PER_REGION = 60
RANDOM_SEED = 42

train_zip_path = Path("/content/Train.zip")

train_sample_root = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/train_samples"
)


def get_matching_file_names(zip_file, region):
    """
    ZIP içerisinde hem görüntüsü hem maskesi bulunan
    PNG dosyalarının ortak isimlerini döndürür.
    """

    image_prefix = f"Train/{region}/images_png/"
    mask_prefix = f"Train/{region}/masks_png/"

    image_files = {
        Path(path).name
        for path in zip_file.namelist()
        if path.startswith(image_prefix)
        and path.endswith(".png")
    }

    mask_files = {
        Path(path).name
        for path in zip_file.namelist()
        if path.startswith(mask_prefix)
        and path.endswith(".png")
    }

    return sorted(image_files & mask_files)


def extract_selected_pairs(
    zip_file,
    region,
    selected_file_names,
    destination_root
):
    """
    Seçilen görüntü ve maskeleri ZIP içerisinden
    doğrudan Google Drive'a çıkarır.
    """

    image_output_dir = (
        destination_root / region / "images"
    )

    mask_output_dir = (
        destination_root / region / "masks"
    )

    image_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    mask_output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for file_name in selected_file_names:

        archive_image_path = (
            f"Train/{region}/images_png/{file_name}"
        )

        archive_mask_path = (
            f"Train/{region}/masks_png/{file_name}"
        )

        output_image_path = (
            image_output_dir / file_name
        )

        output_mask_path = (
            mask_output_dir / file_name
        )

        # Görüntüyü ZIP'ten Drive'a kopyala
        with zip_file.open(
            archive_image_path
        ) as source_file:

            with open(
                output_image_path,
                "wb"
            ) as destination_file:

                shutil.copyfileobj(
                    source_file,
                    destination_file
                )

        # Maskeyi ZIP'ten Drive'a kopyala
        with zip_file.open(
            archive_mask_path
        ) as source_file:

            with open(
                output_mask_path,
                "wb"
            ) as destination_file:

                shutil.copyfileobj(
                    source_file,
                    destination_file
                )


random.seed(RANDOM_SEED)

with zipfile.ZipFile(
    train_zip_path,
    "r"
) as zip_file:

    urban_matching_files = get_matching_file_names(
        zip_file,
        "Urban"
    )

    rural_matching_files = get_matching_file_names(
        zip_file,
        "Rural"
    )

    print(
        "ZIP içerisindeki Urban çift sayısı:",
        len(urban_matching_files)
    )

    print(
        "ZIP içerisindeki Rural çift sayısı:",
        len(rural_matching_files)
    )

    if len(urban_matching_files) < TRAIN_SAMPLE_COUNT_PER_REGION:
        raise ValueError(
            "Yeterli Urban görüntü–maske çifti bulunamadı."
        )

    if len(rural_matching_files) < TRAIN_SAMPLE_COUNT_PER_REGION:
        raise ValueError(
            "Yeterli Rural görüntü–maske çifti bulunamadı."
        )

    selected_urban_files = random.sample(
        urban_matching_files,
        TRAIN_SAMPLE_COUNT_PER_REGION
    )

    selected_rural_files = random.sample(
        rural_matching_files,
        TRAIN_SAMPLE_COUNT_PER_REGION
    )

    print("\nUrban örnekleri çıkarılıyor...")

    extract_selected_pairs(
        zip_file,
        "Urban",
        selected_urban_files,
        train_sample_root
    )

    print("Rural örnekleri çıkarılıyor...")

    extract_selected_pairs(
        zip_file,
        "Rural",
        selected_rural_files,
        train_sample_root
    )


# Seçilen dosyaları kaydediyoruz.
# Böylece ileride aynı eğitim örneklerini tekrar kullanabiliriz.
selection_information = {
    "random_seed": RANDOM_SEED,
    "urban_files": sorted(selected_urban_files),
    "rural_files": sorted(selected_rural_files)
}

selection_file_path = (
    train_sample_root / "selected_files.json"
)

with open(
    selection_file_path,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        selection_information,
        json_file,
        ensure_ascii=False,
        indent=4
    )


print("\nEğitim örnekleri başarıyla oluşturuldu.")

print(
    "Urban görüntü–maske çifti:",
    len(selected_urban_files)
)

print(
    "Rural görüntü–maske çifti:",
    len(selected_rural_files)
)

print(
    "Toplam eğitim çifti:",
    len(selected_urban_files)
    + len(selected_rural_files)
)

print("\nKayıt konumu:")
print(train_sample_root)

ZIP içerisindeki Urban çift sayısı: 1156
ZIP içerisindeki Rural çift sayısı: 1366

Urban örnekleri çıkarılıyor...
Rural örnekleri çıkarılıyor...

Eğitim örnekleri başarıyla oluşturuldu.
Urban görüntü–maske çifti: 60
Rural görüntü–maske çifti: 60
Toplam eğitim çifti: 120

Kayıt konumu:
/content/drive/MyDrive/smart-city-satellite-analysis/train_samples


In [ ]:
from pathlib import Path

train_sample_root = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/train_samples"
)

urban_train_image_dir = (
    train_sample_root / "Urban" / "images"
)

urban_train_mask_dir = (
    train_sample_root / "Urban" / "masks"
)

rural_train_image_dir = (
    train_sample_root / "Rural" / "images"
)

rural_train_mask_dir = (
    train_sample_root / "Rural" / "masks"
)


urban_image_files = sorted(
    urban_train_image_dir.glob("*.png")
)

urban_mask_files = sorted(
    urban_train_mask_dir.glob("*.png")
)

rural_image_files = sorted(
    rural_train_image_dir.glob("*.png")
)

rural_mask_files = sorted(
    rural_train_mask_dir.glob("*.png")
)


print("Urban görüntü sayısı:", len(urban_image_files))
print("Urban maske sayısı:", len(urban_mask_files))

print("Rural görüntü sayısı:", len(rural_image_files))
print("Rural maske sayısı:", len(rural_mask_files))

print(
    "\nToplam görüntü–maske çifti:",
    len(urban_image_files)
    + len(rural_image_files)
)

Urban görüntü sayısı: 60
Urban maske sayısı: 60
Rural görüntü sayısı: 60
Rural maske sayısı: 60

Toplam görüntü–maske çifti: 120


## Eğitim ve Validation Veri Yükleyicilerinin Hazırlanması

LoveDA Train bölümünden seçilen 120 görüntü–maske çifti modelin eğitilmesi için kullanılacaktır.

LoveDA Validation bölümünden daha önce seçilen 40 görüntü–maske çifti ise modelin eğitim sırasında görmediği veriler üzerindeki başarısını ölçmek için kullanılacaktır.

- Training DataLoader: Model ağırlıklarının güncellenmesinde kullanılır.
- Validation DataLoader: Model ağırlıklarını güncellemez, yalnızca performansı ölçer.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset


# LoveDA'daki 0 değeri no-data alanıdır.
# Eğitim sırasında bu pikseller dikkate alınmayacak.
IGNORE_INDEX = 255

# Modelimizin öğreneceği gerçek sınıf sayısı
NUM_CLASSES = 7

model_class_names = {
    0: "Background",
    1: "Building",
    2: "Road",
    3: "Water",
    4: "Barren",
    5: "Forest",
    6: "Agriculture"
}


def remap_loveda_mask(mask_array):
    """
    LoveDA maskesini modelin kullanacağı sınıf düzenine dönüştürür.

    LoveDA sınıfları:
        0   -> no-data
        1-7 -> gerçek sınıflar

    Model sınıfları:
        0-6   -> gerçek sınıflar
        255   -> eğitimde yok sayılacak pikseller
    """

    remapped_mask = np.full(
        mask_array.shape,
        IGNORE_INDEX,
        dtype=np.uint8
    )

    valid_pixels = mask_array > 0

    remapped_mask[valid_pixels] = (
        mask_array[valid_pixels] - 1
    )

    return remapped_mask


class LoveDADataset(Dataset):

    def __init__(
        self,
        image_directory,
        mask_directory,
        file_names,
        target_size=(256, 256)
    ):
        self.image_directory = Path(image_directory)
        self.mask_directory = Path(mask_directory)
        self.file_names = list(file_names)
        self.target_size = target_size

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, index):

        file_name = self.file_names[index]

        image_path = (
            self.image_directory / file_name
        )

        mask_path = (
            self.mask_directory / file_name
        )

        # Uydu görüntüsünü RGB olarak aç
        image = Image.open(
            image_path
        ).convert("RGB")

        # Maskeyi sınıf numaraları korunacak şekilde aç
        mask = Image.open(mask_path)

        # Uydu görüntüsünde yumuşak yeniden boyutlandırma
        image = image.resize(
            self.target_size,
            resample=Image.Resampling.BILINEAR
        )

        # Maskede sınıf numaralarını koruyan yeniden boyutlandırma
        mask = mask.resize(
            self.target_size,
            resample=Image.Resampling.NEAREST
        )

        image_array = np.array(
            image,
            dtype=np.float32
        )

        mask_array = np.array(
            mask,
            dtype=np.uint8
        )

        # LoveDA 1-7 sınıflarını model için 0-6 yap
        mask_array = remap_loveda_mask(
            mask_array
        )

        # Görüntü:
        # [yükseklik, genişlik, kanal]
        #              ↓
        # [kanal, yükseklik, genişlik]
        image_tensor = torch.from_numpy(
            image_array
        ).permute(2, 0, 1)

        # Piksel değerlerini 0-255 aralığından 0-1 aralığına getir
        image_tensor = image_tensor / 255.0

        # CrossEntropyLoss maskeyi int64 bekler
        mask_tensor = torch.from_numpy(
            mask_array.astype(np.int64)
        )

        return image_tensor, mask_tensor


print("Dataset sınıfı başarıyla oluşturuldu.")
print("Sınıf sayısı:", NUM_CLASSES)
print("Yok sayılacak maske değeri:", IGNORE_INDEX)

Dataset sınıfı başarıyla oluşturuldu.
Sınıf sayısı: 7
Yok sayılacak maske değeri: 255


In [ ]:
# Eğitim verileri:
# LoveDA Train bölümünden seçtiğimiz 120 çift

train_sample_root = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/train_samples"
)

urban_train_image_dir = (
    train_sample_root / "Urban" / "images"
)

urban_train_mask_dir = (
    train_sample_root / "Urban" / "masks"
)

rural_train_image_dir = (
    train_sample_root / "Rural" / "images"
)

rural_train_mask_dir = (
    train_sample_root / "Rural" / "masks"
)


# Validation verileri:
# LoveDA Val bölümünden daha önce seçtiğimiz 40 çift

validation_sample_root = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/data_samples"
)

urban_validation_image_dir = (
    validation_sample_root / "Urban" / "images"
)

urban_validation_mask_dir = (
    validation_sample_root / "Urban" / "masks"
)

rural_validation_image_dir = (
    validation_sample_root / "Rural" / "images"
)

rural_validation_mask_dir = (
    validation_sample_root / "Rural" / "masks"
)


directories = {
    "Urban train images": urban_train_image_dir,
    "Urban train masks": urban_train_mask_dir,
    "Rural train images": rural_train_image_dir,
    "Rural train masks": rural_train_mask_dir,
    "Urban validation images": urban_validation_image_dir,
    "Urban validation masks": urban_validation_mask_dir,
    "Rural validation images": rural_validation_image_dir,
    "Rural validation masks": rural_validation_mask_dir
}


for directory_name, directory_path in directories.items():
    print(
        f"{directory_name:<30}",
        directory_path.exists()
    )

Urban train images             True
Urban train masks              True
Rural train images             True
Rural train masks              True
Urban validation images        True
Urban validation masks         True
Rural validation images        True
Rural validation masks         True


In [ ]:
def find_matching_files(
    image_directory,
    mask_directory
):
    """
    Hem görüntüsü hem maskesi bulunan PNG dosyalarını döndürür.
    """

    image_files = {
        file_path.name
        for file_path in Path(
            image_directory
        ).glob("*.png")
    }

    mask_files = {
        file_path.name
        for file_path in Path(
            mask_directory
        ).glob("*.png")
    }

    return sorted(
        image_files & mask_files
    )


urban_train_files = find_matching_files(
    urban_train_image_dir,
    urban_train_mask_dir
)

rural_train_files = find_matching_files(
    rural_train_image_dir,
    rural_train_mask_dir
)

urban_validation_files = find_matching_files(
    urban_validation_image_dir,
    urban_validation_mask_dir
)

rural_validation_files = find_matching_files(
    rural_validation_image_dir,
    rural_validation_mask_dir
)


print("Urban train çift sayısı:", len(urban_train_files))
print("Rural train çift sayısı:", len(rural_train_files))

print(
    "Toplam train çift sayısı:",
    len(urban_train_files)
    + len(rural_train_files)
)

print()

print(
    "Urban validation çift sayısı:",
    len(urban_validation_files)
)

print(
    "Rural validation çift sayısı:",
    len(rural_validation_files)
)

print(
    "Toplam validation çift sayısı:",
    len(urban_validation_files)
    + len(rural_validation_files)
)

Urban train çift sayısı: 60
Rural train çift sayısı: 60
Toplam train çift sayısı: 120

Urban validation çift sayısı: 20
Rural validation çift sayısı: 20
Toplam validation çift sayısı: 40


In [ ]:
urban_train_dataset = LoveDADataset(
    image_directory=urban_train_image_dir,
    mask_directory=urban_train_mask_dir,
    file_names=urban_train_files,
    target_size=(256, 256)
)

rural_train_dataset = LoveDADataset(
    image_directory=rural_train_image_dir,
    mask_directory=rural_train_mask_dir,
    file_names=rural_train_files,
    target_size=(256, 256)
)

urban_validation_dataset = LoveDADataset(
    image_directory=urban_validation_image_dir,
    mask_directory=urban_validation_mask_dir,
    file_names=urban_validation_files,
    target_size=(256, 256)
)

rural_validation_dataset = LoveDADataset(
    image_directory=rural_validation_image_dir,
    mask_directory=rural_validation_mask_dir,
    file_names=rural_validation_files,
    target_size=(256, 256)
)


print(
    "Urban train Dataset:",
    len(urban_train_dataset)
)

print(
    "Rural train Dataset:",
    len(rural_train_dataset)
)

print(
    "Urban validation Dataset:",
    len(urban_validation_dataset)
)

print(
    "Rural validation Dataset:",
    len(rural_validation_dataset)
)

Urban train Dataset: 60
Rural train Dataset: 60
Urban validation Dataset: 20
Rural validation Dataset: 20


In [ ]:
from torch.utils.data import ConcatDataset


train_dataset = ConcatDataset([
    urban_train_dataset,
    rural_train_dataset
])

validation_dataset = ConcatDataset([
    urban_validation_dataset,
    rural_validation_dataset
])


print(
    "Birleştirilmiş train Dataset:",
    len(train_dataset)
)

print(
    "Birleştirilmiş validation Dataset:",
    len(validation_dataset)
)

Birleştirilmiş train Dataset: 120
Birleştirilmiş validation Dataset: 40


In [ ]:
from torch.utils.data import DataLoader


BATCH_SIZE = 4

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Batch boyutu:", BATCH_SIZE)

print(
    "Train batch sayısı:",
    len(train_dataloader)
)

print(
    "Validation batch sayısı:",
    len(validation_dataloader)
)

Batch boyutu: 4
Train batch sayısı: 30
Validation batch sayısı: 10


In [ ]:
train_images, train_masks = next(
    iter(train_dataloader)
)

validation_images, validation_masks = next(
    iter(validation_dataloader)
)


print("Train görüntü batch boyutu:")
print(train_images.shape)

print("\nTrain maske batch boyutu:")
print(train_masks.shape)

print("\nValidation görüntü batch boyutu:")
print(validation_images.shape)

print("\nValidation maske batch boyutu:")
print(validation_masks.shape)

print("\nTrain maskelerindeki sınıflar:")
print(torch.unique(train_masks))

print("\nValidation maskelerindeki sınıflar:")
print(torch.unique(validation_masks))

Train görüntü batch boyutu:
torch.Size([4, 3, 256, 256])

Train maske batch boyutu:
torch.Size([4, 256, 256])

Validation görüntü batch boyutu:
torch.Size([4, 3, 256, 256])

Validation maske batch boyutu:
torch.Size([4, 256, 256])

Train maskelerindeki sınıflar:
tensor([0, 1, 2, 3, 5, 6])

Validation maskelerindeki sınıflar:
tensor([  0,   1,   2,   3,   4,   5,   6, 255])


In [ ]:
def find_all_mask_values(dataloader):
    all_values = set()

    for _, masks in dataloader:
        values = torch.unique(masks).tolist()
        all_values.update(values)

    return sorted(all_values)


train_mask_values = find_all_mask_values(
    train_dataloader
)

validation_mask_values = find_all_mask_values(
    validation_dataloader
)

print("Tüm train verisindeki maske değerleri:")
print(train_mask_values)

print("\nTüm validation verisindeki maske değerleri:")
print(validation_mask_values)

Tüm train verisindeki maske değerleri:
[0, 1, 2, 3, 4, 5, 6, 255]

Tüm validation verisindeki maske değerleri:
[0, 1, 2, 3, 4, 5, 6, 255]


## U-Net Modelinin Oluşturulması

U-Net, semantic segmentation işlemlerinde kullanılan encoder-decoder tabanlı bir sinir ağıdır.

- Encoder, görüntüden özellik çıkarır ve uzamsal boyutu küçültür.
- Bottleneck, modelin en yoğun özellik temsilini oluşturur.
- Decoder, özellik haritalarını tekrar görüntü boyutuna büyütür.
- Skip connection bağlantıları, encoder sırasında elde edilen konumsal ayrıntıları decoder katmanlarına aktarır.

Modelin son katmanı her piksel için yedi arazi sınıfına ait ayrı bir skor üretecektir.

In [ ]:
import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    """
    Arka arkaya iki convolution işlemi uygular.

    Her convolution sonrasında:
    - Batch Normalization
    - ReLU aktivasyonu

    kullanılır.
    """

    def __init__(
        self,
        input_channels,
        output_channels
    ):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(
                input_channels,
                output_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class UNet(nn.Module):

    def __init__(
        self,
        input_channels=3,
        num_classes=7
    ):
        super().__init__()

        # Encoder
        self.encoder1 = DoubleConv(
            input_channels,
            32
        )

        self.pool1 = nn.MaxPool2d(2)

        self.encoder2 = DoubleConv(
            32,
            64
        )

        self.pool2 = nn.MaxPool2d(2)

        self.encoder3 = DoubleConv(
            64,
            128
        )

        self.pool3 = nn.MaxPool2d(2)

        self.encoder4 = DoubleConv(
            128,
            256
        )

        self.pool4 = nn.MaxPool2d(2)

        # U-Net'in en alt bölümü
        self.bottleneck = DoubleConv(
            256,
            512
        )

        # Decoder
        self.upconv4 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.decoder4 = DoubleConv(
            512,
            256
        )

        self.upconv3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.decoder3 = DoubleConv(
            256,
            128
        )

        self.upconv2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.decoder2 = DoubleConv(
            128,
            64
        )

        self.upconv1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.decoder1 = DoubleConv(
            64,
            32
        )

        # Her piksel için 7 sınıf skoru üretir
        self.output_layer = nn.Conv2d(
            32,
            num_classes,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        encoder1_output = self.encoder1(x)

        encoder2_output = self.encoder2(
            self.pool1(encoder1_output)
        )

        encoder3_output = self.encoder3(
            self.pool2(encoder2_output)
        )

        encoder4_output = self.encoder4(
            self.pool3(encoder3_output)
        )

        bottleneck_output = self.bottleneck(
            self.pool4(encoder4_output)
        )

        # Decoder 4
        decoder4_output = self.upconv4(
            bottleneck_output
        )

        decoder4_output = torch.cat(
            [
                decoder4_output,
                encoder4_output
            ],
            dim=1
        )

        decoder4_output = self.decoder4(
            decoder4_output
        )

        # Decoder 3
        decoder3_output = self.upconv3(
            decoder4_output
        )

        decoder3_output = torch.cat(
            [
                decoder3_output,
                encoder3_output
            ],
            dim=1
        )

        decoder3_output = self.decoder3(
            decoder3_output
        )

        # Decoder 2
        decoder2_output = self.upconv2(
            decoder3_output
        )

        decoder2_output = torch.cat(
            [
                decoder2_output,
                encoder2_output
            ],
            dim=1
        )

        decoder2_output = self.decoder2(
            decoder2_output
        )

        # Decoder 1
        decoder1_output = self.upconv1(
            decoder2_output
        )

        decoder1_output = torch.cat(
            [
                decoder1_output,
                encoder1_output
            ],
            dim=1
        )

        decoder1_output = self.decoder1(
            decoder1_output
        )

        logits = self.output_layer(
            decoder1_output
        )

        return logits

In [ ]:
model = UNet(
    input_channels=3,
    num_classes=NUM_CLASSES
).to(device)

print("Model başarıyla oluşturuldu.")
print("Modelin bulunduğu cihaz:", device)

Model başarıyla oluşturuldu.
Modelin bulunduğu cihaz: cpu


In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Toplam parametre sayısı:", total_parameters)
print("Öğrenilebilir parametre sayısı:", trainable_parameters)

Toplam parametre sayısı: 7763239
Öğrenilebilir parametre sayısı: 7763239


In [ ]:
test_images = train_images.to(device)

with torch.no_grad():
    test_outputs = model(test_images)

print("Modele verilen görüntülerin boyutu:")
print(test_images.shape)

print("\nModel çıktısının boyutu:")
print(test_outputs.shape)

Modele verilen görüntülerin boyutu:
torch.Size([4, 3, 256, 256])

Model çıktısının boyutu:
torch.Size([4, 7, 256, 256])


In [ ]:
test_predictions = torch.argmax(
    test_outputs,
    dim=1
)

print("Tahmin maskelerinin boyutu:")
print(test_predictions.shape)

print("\nTahmin edilen sınıflar:")
print(torch.unique(test_predictions))

Tahmin maskelerinin boyutu:
torch.Size([4, 256, 256])

Tahmin edilen sınıflar:
tensor([0, 1, 2, 3, 4, 5, 6])


## Loss Fonksiyonu ve Optimizasyon

Modelin ürettiği tahminler gerçek segmentation maskeleriyle karşılaştırılacaktır.

CrossEntropyLoss, her piksel için modelin ürettiği yedi sınıf skorunu gerçek sınıf numarasıyla karşılaştırır.

LoveDA veri setindeki no-data pikselleri 255 değerine dönüştürüldüğü için bu pikseller loss hesabında dikkate alınmayacaktır.

Adam optimizasyon algoritması, hesaplanan hataya göre model ağırlıklarını güncelleyecektir.

In [ ]:
import torch.optim as optim

# Her pikselde tahmin edilen sınıf ile gerçek sınıfı karşılaştırır.
# 255 değerindeki no-data piksellerini hesaba katmaz.
criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

# Model ağırlıklarını loss değerine göre günceller.
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss fonksiyonu:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

Loss fonksiyonu: CrossEntropyLoss()
Optimizer: Adam
Learning rate: 0.001


In [ ]:
# Modeli eğitim moduna geçiriyoruz
model.train()

# Train DataLoader'dan ilk batch'i alıyoruz
batch_images, batch_masks = next(
    iter(train_dataloader)
)

# Verileri modelin bulunduğu cihaza taşıyoruz
batch_images = batch_images.to(device)
batch_masks = batch_masks.to(device)

# Önceki gradient değerlerini temizliyoruz
optimizer.zero_grad()

# İleri geçiş:
# Model görüntülerden sınıf skorları üretir
batch_outputs = model(batch_images)

# Tahminlerle gerçek maskeler arasındaki hatayı hesaplıyoruz
batch_loss = criterion(
    batch_outputs,
    batch_masks
)

# Geri yayılım:
# Her parametrenin hataya ne kadar katkı verdiğini hesaplar
batch_loss.backward()

# Model ağırlıklarını günceller
optimizer.step()

print("Görüntü batch boyutu:", batch_images.shape)
print("Model çıktı boyutu:", batch_outputs.shape)
print("Gerçek maske boyutu:", batch_masks.shape)
print(f"İlk eğitim adımındaki loss: {batch_loss.item():.4f}")

Görüntü batch boyutu: torch.Size([4, 3, 256, 256])
Model çıktı boyutu: torch.Size([4, 7, 256, 256])
Gerçek maske boyutu: torch.Size([4, 256, 256])
İlk eğitim adımındaki loss: 2.0670


## Modelin Tam Eğitim Süreci

Model, eğitim verisindeki bütün batch'ler üzerinde birden fazla epoch boyunca eğitilecektir.

Her epoch içerisinde:

1. Model 120 eğitim görüntüsünü görür.
2. Her batch için loss hesaplanır.
3. Geri yayılım ile model ağırlıkları güncellenir.
4. Eğitim tamamlandıktan sonra 40 validation görüntüsü üzerinde loss hesaplanır.
5. Validation loss değeri en düşük olan model Google Drive'a kaydedilir.

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# Sonuçların tekrar üretilebilir olması için
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# Modeli yeniden oluşturuyoruz.
# Böylece daha önce yaptığımız tek batch güncellemesi sıfırlanır.
model = UNet(
    input_channels=3,
    num_classes=NUM_CLASSES
).to(device)


criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)


optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# En iyi modelin kaydedileceği klasör
model_output_directory = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/models"
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    model_output_directory / "best_unet_model.pth"
)


print("Model yeniden oluşturuldu.")
print("Kullanılan cihaz:", device)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Model kayıt konumu:", best_model_path)

Model yeniden oluşturuldu.
Kullanılan cihaz: cpu
Learning rate: 0.001
Model kayıt konumu: /content/drive/MyDrive/smart-city-satellite-analysis/models/best_unet_model.pth


In [ ]:
def validate_one_epoch(
    model,
    dataloader,
    criterion,
    device
):
    """
    Modeli validation verisi üzerinde değerlendirir.

    Bu fonksiyonda geri yayılım yapılmaz ve
    model ağırlıkları değiştirilmez.
    """

    # Model değerlendirme moduna geçer
    model.eval()

    total_loss = 0.0

    # Gradient hesaplamaya gerek yok
    with torch.no_grad():

        for batch_images, batch_masks in dataloader:

            batch_images = batch_images.to(
                device,
                non_blocking=True
            )

            batch_masks = batch_masks.to(
                device,
                non_blocking=True
            )

            batch_outputs = model(batch_images)

            loss = criterion(
                batch_outputs,
                batch_masks
            )

            total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    return average_loss

In [ ]:
def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):
    """
    Modeli bütün eğitim batch'leri üzerinde bir epoch eğitir.
    Ortalama training loss değerini döndürür.
    """

    model.train()

    total_loss = 0.0

    for batch_images, batch_masks in dataloader:

        batch_images = batch_images.to(
            device,
            non_blocking=True
        )

        batch_masks = batch_masks.to(
            device,
            non_blocking=True
        )

        # Önceki batch'ten kalan gradientleri temizle
        optimizer.zero_grad()

        # İleri geçiş
        batch_outputs = model(batch_images)

        # Hatayı hesapla
        loss = criterion(
            batch_outputs,
            batch_masks
        )

        # Geri yayılım
        loss.backward()

        # Ağırlıkları güncelle
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    return average_loss


def validate_one_epoch(
    model,
    dataloader,
    criterion,
    device
):
    """
    Modeli validation verisi üzerinde değerlendirir.
    Model ağırlıkları değiştirilmez.
    """

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for batch_images, batch_masks in dataloader:

            batch_images = batch_images.to(
                device,
                non_blocking=True
            )

            batch_masks = batch_masks.to(
                device,
                non_blocking=True
            )

            batch_outputs = model(batch_images)

            loss = criterion(
                batch_outputs,
                batch_masks
            )

            total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    return average_loss


print("Eğitim ve validation fonksiyonları tanımlandı.")

Eğitim ve validation fonksiyonları tanımlandı.


In [ ]:
NUM_EPOCHS = 10

training_loss_history = []
validation_loss_history = []

best_validation_loss = float("inf")


for epoch in range(NUM_EPOCHS):

    training_loss = train_one_epoch(
        model=model,
        dataloader=train_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    validation_loss = validate_one_epoch(
        model=model,
        dataloader=validation_dataloader,
        criterion=criterion,
        device=device
    )

    training_loss_history.append(
        training_loss
    )

    validation_loss_history.append(
        validation_loss
    )

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {training_loss:.4f} | "
        f"Validation Loss: {validation_loss:.4f}"
    )

    # Validation başarısı iyileştiyse modeli kaydet
    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_loss": validation_loss,
                "class_names": model_class_names,
                "num_classes": NUM_CLASSES,
                "ignore_index": IGNORE_INDEX,
                "target_size": (256, 256)
            },
            best_model_path
        )

        print(
            "  → En iyi model kaydedildi."
        )


print("\nEğitim tamamlandı.")
print(
    f"En iyi validation loss: "
    f"{best_validation_loss:.4f}"
)
print("Model kayıt konumu:")
print(best_model_path)

Epoch 01/10 | Train Loss: 1.2749 | Validation Loss: 1.4689
  → En iyi model kaydedildi.
Epoch 02/10 | Train Loss: 1.2834 | Validation Loss: 1.6595
Epoch 03/10 | Train Loss: 1.2751 | Validation Loss: 1.6218
Epoch 04/10 | Train Loss: 1.2984 | Validation Loss: 1.5362
Epoch 05/10 | Train Loss: 1.2636 | Validation Loss: 1.7155
Epoch 06/10 | Train Loss: 1.2491 | Validation Loss: 1.5870
Epoch 07/10 | Train Loss: 1.2505 | Validation Loss: 1.4581
  → En iyi model kaydedildi.
Epoch 08/10 | Train Loss: 1.2354 | Validation Loss: 1.5610
Epoch 09/10 | Train Loss: 1.2330 | Validation Loss: 1.5459
Epoch 10/10 | Train Loss: 1.2957 | Validation Loss: 1.7516

Eğitim tamamlandı.
En iyi validation loss: 1.4581
Model kayıt konumu:
/content/drive/MyDrive/smart-city-satellite-analysis/models/best_unet_model.pth


In [ ]:
from pathlib import Path

saved_model_path = Path(
    "/content/drive/MyDrive/"
    "smart-city-satellite-analysis/models/"
    "best_unet_model.pth"
)

if saved_model_path.exists():
    model_size_mb = saved_model_path.stat().st_size / (1024 ** 2)

    print("Model güvenli şekilde kaydedilmiş.")
    print(f"Model boyutu: {model_size_mb:.2f} MB")
    print("Konum:", saved_model_path)
else:
    print("Henüz kayıtlı model bulunamadı.")

Model güvenli şekilde kaydedilmiş.
Model boyutu: 88.96 MB
Konum: /content/drive/MyDrive/smart-city-satellite-analysis/models/best_unet_model.pth
